# Installs and Environment Variables

In [ ]:
!pip install -q numpy pillow torch sentence-transformers scipy pymilvus langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
# Colab Secrets
import os
from google.colab import userdata
# For Milvus deployment
os.environ["ZILLIZ_CLOUD_URI"] = userdata.get('ZILLIZ_CLOUD_URI')
os.environ["ZILLIZ_CLOUD_API_KEY"] = userdata.get('ZILLIZ_CLOUD_API_KEY')
os.environ["MILVUS_URI"] = userdata.get('ZILLIZ_CLOUD_URI')
os.environ["MILVUS_TOKEN"] = userdata.get('ZILLIZ_CLOUD_API_KEY')
# For OpenAI LLM
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

# Configuration & Setup

In [ ]:
# ============================================================================
# SECTION A: CONFIGURATION & SETUP
# ============================================================================
print("="*80)
print("PRODUCTION RAG - INITIALIZATION")
print("="*80)

import os
import sys
from typing import List, Dict, Optional, Tuple
import numpy as np
from PIL import Image
import torch
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix
from pymilvus import connections, Collection
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

print("✓ All imports successful")
print()


PRODUCTION RAG - INITIALIZATION
✓ All imports successful



## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Centralized configuration management."""

    # Milvus Configuration
    MILVUS_HOST = os.getenv('MILVUS_HOST', 'localhost')
    MILVUS_PORT = os.getenv('MILVUS_PORT', '19530')
    MILVUS_URI = os.getenv('MILVUS_URI')  # For Zilliz Cloud
    MILVUS_TOKEN = os.getenv('MILVUS_TOKEN')  # For Zilliz Cloud
    COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'product_hybrid_search_mvp2')

    # Model Configuration
    CLIP_MODEL = 'clip-ViT-B-32'
    SPLADE_MODEL = 'naver/splade-cocondenser-ensembledistil'
    SPLADE_TOP_K = 256  # Keep top 256 dimensions for sparse vectors

    # LLM Configuration
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')
    LLM_TEMPERATURE = float(os.getenv('LLM_TEMPERATURE', '0.7'))

    # LangSmith Configuration (optional)
    LANGCHAIN_TRACING_V2 = os.getenv('LANGCHAIN_TRACING_V2', 'false')
    LANGCHAIN_PROJECT = os.getenv('LANGCHAIN_PROJECT', 'prod-rag')
    LANGCHAIN_API_KEY = os.getenv('LANGCHAIN_API_KEY')

    # Search Configuration
    DEFAULT_TOP_K = int(os.getenv('DEFAULT_TOP_K', '5'))
    RRF_K = int(os.getenv('RRF_K', '60'))
    TEXT_WEIGHT = float(os.getenv('TEXT_WEIGHT', '0.6'))

    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    @classmethod
    def validate(cls):
        """Validate required configuration."""
        errors = []

        if not cls.MILVUS_URI and not cls.MILVUS_HOST:
            errors.append("MILVUS_HOST or MILVUS_URI must be set")

        if not cls.OPENAI_API_KEY:
            errors.append("OPENAI_API_KEY must be set")

        if errors:
            raise ValueError(f"Configuration errors: {', '.join(errors)}")

    @classmethod
    def print_config(cls):
        """Print current configuration (hide sensitive values)."""
        print("Current Configuration:")
        print(f"  Milvus: {cls.MILVUS_URI or f'{cls.MILVUS_HOST}:{cls.MILVUS_PORT}'}")
        print(f"  Collection: {cls.COLLECTION_NAME}")
        print(f"  CLIP Model: {cls.CLIP_MODEL}")
        print(f"  LLM Model: {cls.LLM_MODEL}")
        print(f"  Device: {cls.DEVICE}")
        print(f"  LangSmith Tracing: {cls.LANGCHAIN_TRACING_V2}")
        print()

# Validate configuration
try:
    Config.validate()
    Config.print_config()
except ValueError as e:
    print(f"❌ Configuration Error: {e}")
    print("\nPlease set required environment variables:")
    print("  export OPENAI_API_KEY='your-key'")
    print("  export MILVUS_URI='your-zilliz-uri'  # or use MILVUS_HOST")
    sys.exit(1)


Current Configuration:
  Milvus: https://in03-8905c4d2f7847b8.serverless.gcp-us-west1.cloud.zilliz.com
  Collection: product_hybrid_search_mvp2
  CLIP Model: clip-ViT-B-32
  LLM Model: gpt-4o-mini
  Device: cpu
  LangSmith Tracing: false



## Connect to Milvus

In [ ]:
# ============================================================================
# SECTION A1: CONNECT TO MILVUS
# ============================================================================
print("Connecting to Milvus...")

def connect_to_milvus() -> Collection:
    """Connect to Milvus and return collection."""
    try:
        if Config.MILVUS_URI:
            # Zilliz Cloud
            connections.connect(
                alias="default",
                uri=Config.MILVUS_URI,
                token=Config.MILVUS_TOKEN
            )
        else:
            # Local Milvus
            connections.connect(
                alias="default",
                host=Config.MILVUS_HOST,
                port=Config.MILVUS_PORT
            )

        collection = Collection(Config.COLLECTION_NAME)
        collection.load()

        print(f"✓ Connected to Milvus collection: {Config.COLLECTION_NAME}")
        print(f"✓ Collection entities: {collection.num_entities}")
        return collection

    except Exception as e:
        print(f"❌ Failed to connect to Milvus: {e}")
        sys.exit(1)

collection = connect_to_milvus()
print()

Connecting to Milvus...
✓ Connected to Milvus collection: product_hybrid_search_mvp2
✓ Collection entities: 9974



## Load Models

In [ ]:
# ============================================================================
# SECTION A2: LOAD MODELS
# ============================================================================
print("Loading AI models (this may take 10-30 seconds on first load)...")

def load_clip_model() -> SentenceTransformer:
    """Load CLIP model for dense embeddings."""
    try:
        model = SentenceTransformer(Config.CLIP_MODEL, device=Config.DEVICE)
        model.eval()
        torch.set_num_threads(2)
        print(f"✓ CLIP model loaded on {Config.DEVICE}")
        return model
    except Exception as e:
        print(f"❌ Failed to load CLIP: {e}")
        sys.exit(1)

def load_splade_model() -> Optional[SentenceTransformer]:
    """Load SPLADE model for sparse embeddings."""
    try:
        model = SentenceTransformer(Config.SPLADE_MODEL, device=Config.DEVICE)
        model.eval()
        print(f"✓ SPLADE model loaded on {Config.DEVICE}")
        return model
    except Exception as e:
        print(f"⚠ SPLADE model not available: {e}")
        print("⚠ Continuing with CLIP-only mode")
        return None

def load_llm() -> ChatOpenAI:
    """Load LLM for generation."""
    try:
        llm = ChatOpenAI(
            model=Config.LLM_MODEL,
            temperature=Config.LLM_TEMPERATURE,
            api_key=Config.OPENAI_API_KEY
        )
        print(f"✓ LLM initialized: {Config.LLM_MODEL}")
        return llm
    except Exception as e:
        print(f"❌ Failed to initialize LLM: {e}")
        sys.exit(1)

# Load all models
clip_model = load_clip_model()
splade_model = load_splade_model()
llm = load_llm()

print()
print("="*80)
print("✓ INITIALIZATION COMPLETE - READY TO SERVE REQUESTS")
print("="*80)
print()

Loading AI models (this may take 10-30 seconds on first load)...


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✓ SPLADE model loaded on cpu
✓ LLM initialized: gpt-4o-mini

✓ INITIALIZATION COMPLETE - READY TO SERVE REQUESTS



# Retrieval Module

## Helper Functions (Query and Retrieval)

In [ ]:
# ============================================================================
# SECTION B: RETRIEVAL MODULE (STANDALONE)
# ============================================================================

def convert_to_sparse_csr(dense_array: np.ndarray, top_k: int = 256) -> csr_matrix:
    """
    Convert dense array to scipy CSR sparse matrix.

    Args:
        dense_array: Dense numpy array
        top_k: Keep only top K values

    Returns:
        scipy CSR matrix (1 row)
    """
    top_indices = np.argsort(dense_array)[-top_k:]
    top_values = dense_array[top_indices]

    n_cols = len(dense_array)
    sparse_matrix = csr_matrix(
        (top_values, (np.zeros(len(top_indices), dtype=int), top_indices)),
        shape=(1, n_cols)
    )

    return sparse_matrix


def encode_text_query(text: str) -> Tuple[np.ndarray, Optional[csr_matrix]]:
    """
    Encode text query with CLIP (dense) and SPLADE (sparse).

    Args:
        text: Query text

    Returns:
        (dense_embedding, sparse_embedding_csr)
    """
    # CLIP dense encoding
    dense = clip_model.encode([text], convert_to_numpy=True)
    dense = dense / np.linalg.norm(dense)

    # SPLADE sparse encoding
    sparse_csr = None
    if splade_model is not None:
        sparse_array = splade_model.encode([text], convert_to_numpy=True)[0]
        sparse_csr = convert_to_sparse_csr(sparse_array, top_k=Config.SPLADE_TOP_K)

    return dense[0], sparse_csr


def encode_image_query(image: Image.Image) -> np.ndarray:
    """
    Encode image query with CLIP.

    Args:
        image: PIL Image

    Returns:
        dense_embedding
    """
    dense = clip_model.encode([image], convert_to_numpy=True)
    dense = dense / np.linalg.norm(dense)
    return dense[0]


def encode_hybrid_query(
    text: str,
    image: Image.Image,
    text_weight: float = None
) -> Tuple[np.ndarray, Optional[csr_matrix]]:
    """
    Encode hybrid query (text + image) with weighted fusion.

    Args:
        text: Query text
        image: PIL Image
        text_weight: Weight for text (default from Config)

    Returns:
        (fused_dense_embedding, sparse_embedding_csr)
    """
    if text_weight is None:
        text_weight = Config.TEXT_WEIGHT

    text_dense, text_sparse = encode_text_query(text)
    image_dense = encode_image_query(image)

    # Fuse dense embeddings
    fused_dense = text_weight * text_dense + (1 - text_weight) * image_dense
    fused_dense = fused_dense / np.linalg.norm(fused_dense)

    return fused_dense, text_sparse


def reciprocal_rank_fusion(result_lists: List[List], k: int = None) -> List[Dict]:
    """
    Combine multiple result lists using Reciprocal Rank Fusion.

    Args:
        result_lists: List of search result lists
        k: RRF constant (default from Config)

    Returns:
        Combined and re-ranked results
    """
    if k is None:
        k = Config.RRF_K

    combined_scores = {}

    for result_list in result_lists:
        for rank, hit in enumerate(result_list):
            product_id = hit.entity.get('product_id')
            rrf_score = 1.0 / (k + rank + 1)

            if product_id not in combined_scores:
                combined_scores[product_id] = {
                    'entity': hit.entity,
                    'score': 0.0,
                    'sources': []
                }

            combined_scores[product_id]['score'] += rrf_score
            combined_scores[product_id]['sources'].append({
                'rank': rank + 1,
                'distance': hit.distance
            })

    sorted_results = sorted(
        combined_scores.values(),
        key=lambda x: x['score'],
        reverse=True
    )

    return sorted_results


def retrieve_products(
    query_text: Optional[str] = None,
    query_image: Optional[Image.Image] = None,
    top_k: int = None,
    text_weight: float = None
) -> List[Dict]:
    """
    Retrieve products using hybrid search (CLIP + SPLADE).

    This is the main retrieval API - can be used standalone without generation.

    Args:
        query_text: Text query (optional)
        query_image: PIL Image query (optional)
        top_k: Number of results (default from Config)
        text_weight: Weight for text in hybrid queries (default from Config)

    Returns:
        List of product dictionaries with metadata and scores

    Raises:
        ValueError: If neither query_text nor query_image provided

    Example:
        # Text search
        products = retrieve_products("garden stool", top_k=5)

        # Image search
        img = Image.open("stool.jpg")
        products = retrieve_products(query_image=img, top_k=5)

        # Hybrid search
        products = retrieve_products("blue color", img, top_k=5)
    """
    if not query_text and not query_image:
        raise ValueError("Must provide either query_text or query_image")

    if top_k is None:
        top_k = Config.DEFAULT_TOP_K

    # Encode query
    if query_text and query_image:
        dense_emb, sparse_csr = encode_hybrid_query(query_text, query_image, text_weight)
    elif query_text:
        dense_emb, sparse_csr = encode_text_query(query_text)
    else:
        dense_emb = encode_image_query(query_image)
        sparse_csr = None

    # Search all vector fields
    search_params = {"metric_type": "COSINE", "params": {"ef": 100}}
    result_lists = []

    # Search text dense embeddings
    text_dense_results = collection.search(
        data=[dense_emb.tolist()],
        anns_field="text_dense_embedding",
        param=search_params,
        limit=top_k * 2,
        output_fields=[
            "product_id", "product_name", "image_url", "selling_price",
            "clip_text", "about_product"
        ]
    )[0]
    result_lists.append(text_dense_results)

    # Search image dense embeddings
    image_dense_results = collection.search(
        data=[dense_emb.tolist()],
        anns_field="image_dense_embedding",
        param=search_params,
        limit=top_k * 2,
        output_fields=[
            "product_id", "product_name", "image_url", "selling_price",
            "clip_text", "about_product"
        ]
    )[0]
    result_lists.append(image_dense_results)

    # Search sparse embeddings if available
    if sparse_csr is not None and splade_model is not None:
        sparse_search_params = {"metric_type": "IP", "params": {}}
        sparse_results = collection.search(
            data=[sparse_csr],
            anns_field="text_sparse_embedding",
            param=sparse_search_params,
            limit=top_k * 2,
            output_fields=[
                "product_id", "product_name", "image_url", "selling_price",
                "clip_text", "about_product"
            ]
        )[0]
        result_lists.append(sparse_results)

    # Fuse results
    combined_results = reciprocal_rank_fusion(result_lists)

    return combined_results[:top_k]

# Generation Module

## Helper Functions (for LLM Context Generation)

In [ ]:
# ============================================================================
# SECTION C: GENERATION MODULE (OPTIONAL)
# ============================================================================

def create_context_string(products: List[Dict]) -> str:
    """Format products into context string for LLM."""
    if not products:
        return "No relevant products found."

    context_parts = []
    for i, product in enumerate(products, 1):
        entity = product['entity']
        context_parts.append(
            f"[Product {i}]\n"
            f"Name: {entity.get('product_name')}\n"
            f"Price: {entity.get('selling_price')}\n"
            f"Description: {entity.get('about_product', 'N/A')[:300]}\n"
            f"Image URL: {entity.get('image_url')}\n"
        )

    return "\n".join(context_parts)


def create_query_description(query_text: Optional[str], query_image: Optional[Image.Image]) -> str:
    """Create description of query for the prompt."""
    if query_text and query_image:
        return f"Text query: '{query_text}' (User also provided a reference image)"
    elif query_text:
        return f"Text query: '{query_text}'"
    elif query_image:
        return "Visual search query (User provided an image)"
    else:
        return "Query"


# Create prompt template
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful shopping assistant for an e-commerce platform.
Your role is to recommend products based on the user's query and retrieved product information.

Guidelines:
- Be concise and helpful
- Recommend products that best match the user's needs
- Mention key features like price, description, and unique attributes
- If multiple good options exist, explain the differences
- If no good matches found, suggest alternatives or ask clarifying questions
- Do NOT mention image URLs in your response (those are for display purposes)
- Do NOT make up product information - only use what's provided in the context"""),

    ("human", """User Query: {query_description}

Retrieved Products:
{context}

Based on the above products, please provide helpful recommendations to the user.""")
])

# Create generation chain
generation_chain = (
    generation_prompt
    | llm
    | StrOutputParser()
)


def generate_answer(
    query_text: Optional[str],
    query_image: Optional[Image.Image],
    retrieved_products: List[Dict]
) -> str:
    """
    Generate natural language answer from retrieved products.

    Args:
        query_text: User's text query
        query_image: User's image query
        retrieved_products: Products from retrieve_products()

    Returns:
        LLM-generated answer

    Example:
        products = retrieve_products("garden stool", top_k=5)
        answer = generate_answer("garden stool", None, products)
    """
    context = create_context_string(retrieved_products)
    query_description = create_query_description(query_text, query_image)

    answer = generation_chain.invoke({
        "context": context,
        "query_description": query_description
    })

    return answer

## RAGResponse (whole Pipeline in one call)

In [ ]:
class RAGResponse:
    """Response object from RAG pipeline."""

    def __init__(
        self,
        answer: Optional[str],
        products: List[Dict],
        query_type: str,
        query_text: Optional[str] = None,
        query_image: Optional[Image.Image] = None
    ):
        self.answer = answer
        self.products = products
        self.query_type = query_type
        self.query_text = query_text
        self.query_image = query_image

    def to_dict(self) -> dict:
        """Convert to dictionary (useful for API responses)."""
        return {
            'answer': self.answer,
            'products': [
                {
                    'product_name': p['entity'].get('product_name'),
                    'price': p['entity'].get('selling_price'),
                    'image_url': p['entity'].get('image_url'),
                    'score': p['score']
                }
                for p in self.products
            ],
            'query_type': self.query_type
        }


def rag_query(
    query_text: Optional[str] = None,
    query_image: Optional[Image.Image] = None,
    top_k: int = None,
    return_mode: str = "full"
) -> RAGResponse:
    """
    Complete RAG pipeline: retrieve + generate.

    This is the main RAG API - handles text, image, or hybrid queries.

    Args:
        query_text: Text query (optional)
        query_image: PIL Image query (optional)
        top_k: Number of products to retrieve (default from Config)
        return_mode: "full" (with generation) or "retrieval_only"

    Returns:
        RAGResponse object with answer and products

    Raises:
        ValueError: If neither query provided or invalid return_mode

    Example:
        # Full RAG
        response = rag_query("garden stool", return_mode="full")
        print(response.answer)
        for p in response.products:
            print(p['entity']['product_name'])

        # Retrieval only (faster, no LLM call)
        response = rag_query("garden stool", return_mode="retrieval_only")
        products = response.products
    """
    if not query_text and not query_image:
        raise ValueError("Must provide either query_text or query_image")

    if return_mode not in ["full", "retrieval_only"]:
        raise ValueError("return_mode must be 'full' or 'retrieval_only'")

    # Determine query type
    if query_text and query_image:
        query_type = "hybrid"
    elif query_text:
        query_type = "text"
    else:
        query_type = "image"

    # Retrieve products
    products = retrieve_products(
        query_text=query_text,
        query_image=query_image,
        top_k=top_k
    )

    # Generate answer if requested
    answer = None
    if return_mode == "full":
        answer = generate_answer(query_text, query_image, products)

    return RAGResponse(
        answer=answer,
        products=products,
        query_type=query_type,
        query_text=query_text,
        query_image=query_image
    )


# Test Queries and Usage

In [ ]:
print("\n🧪 Running test queries...")

# Test 1: Text query
print("\nTest 1: Text query")
response = rag_query("garden stool", top_k=3, return_mode="full")
print(f"Answer: {response.answer[:200]}...")
print(f"Retrieved {len(response.products)} products")

# Test 2: Retrieval only
print("\nTest 2: Retrieval only (no LLM)")
response = rag_query("puzzle", top_k=5, return_mode="retrieval_only")
print(f"Retrieved {len(response.products)} products")
for p in response.products[:3]:
    print(f"  - {p['entity']['product_name'][:50]}")

print("\n✅ Tests completed successfully")


🧪 Running test queries...

Test 1: Text query
Answer: Here are two garden stool options that might suit your needs:

1. **Stoolimals Collection, Faux Fur Gazelle Stool, Black**
   - **Price:** $113.99
   - **Description:** This stool is part of the Keiki...
Retrieved 3 products

Test 2: Retrieval only (no LLM)
Retrieved 5 products
  - -
  - Disney "Cars 2" Mini Puzzle Cube, Party Favor
  - Springbok Puzzles - Christmas Wishes - 400 Piece J

✅ Tests completed successfully


A few more tests

In [ ]:
response = rag_query("Do you guys have the LEGO Speed Champions Dodge Challenger", top_k=10, return_mode="full")
print(f"Answer: {response.answer[:200]}...")
print(f"Retrieved {len(response.products)} products")

Answer: Yes, we have the LEGO Speed Champions Dodge Challenger available! Here are the details:

**LEGO Speed Champions 2018 Dodge Challenger SRT Demon and 1970 Dodge Charger R/T 75893 Building Kit (2019)**  ...
Retrieved 10 products


In [ ]:
print(f"Answer: {response.answer[:]}")
print(f"Retrieved {len(response.products)} products")

Answer: Yes, we have the LEGO Speed Champions Dodge Challenger available! Here are the details:

**LEGO Speed Champions 2018 Dodge Challenger SRT Demon and 1970 Dodge Charger R/T 75893 Building Kit (2019)**  
- **Price**: $14.99  
- **Description**: This building kit includes 478 pieces and features both the 2018 Dodge Challenger SRT Demon with two sets of wheel rims and a rear spoiler, as well as the classic 1970 Dodge Charger R/T with a removable supercharger. It also includes a buildable drag racing Christmas tree start light and three minifigures to enhance your play experience.

This is the only LEGO Speed Champions product that matches your query. If you're looking for something specific or have any other preferences, feel free to let me know!
Retrieved 10 products


With images

In [ ]:
# Test 3: Image query from URL
print("\nTest 3: Image query from URL")
import requests
from io import BytesIO

image_url = "https://m.media-amazon.com/images/I/81SIAvPac3L._AC_SL1500_.jpg"  # Example product image
response = requests.get(image_url)
test_image = Image.open(BytesIO(response.content))

response = rag_query(query_image=test_image, top_k=3, return_mode="full")
print(f"Answer: {response.answer[:200]}...")
print(f"Retrieved {len(response.products)} products")
for p in response.products[:3]:
    print(f"  - {p['entity']['product_name'][:50]}")


Test 3: Image query from URL
Answer: Based on your visual search, here are some great LEGO building kits that feature iconic cars:

1. **LEGO Speed Champions 2018 Dodge Challenger SRT Demon and 1970 Dodge Charger R/T (75893)**
   - **Pri...
Retrieved 3 products
  - LEGO Speed Champions 2018 Dodge Challenger SRT Dem
  - LEGO Speed Champions 1974 Porsche 911 Turbo 3.0 75
  - Tamiya 20068 1/20 Ferrari SF70H Plastic Model Kit


In [ ]:
print(f"Answer: {response.answer[:]}...")
print(f"Retrieved {len(response.products)} products")
for p in response.products[:3]:
    print(f"  - {p['entity']['product_name'][:50]}")

Answer: Based on your visual search, here are some great LEGO building kits that feature iconic cars:

1. **LEGO Speed Champions 2018 Dodge Challenger SRT Demon and 1970 Dodge Charger R/T (75893)**
   - **Price:** $14.99
   - **Description:** This set includes 478 pieces and allows you to build two classic American muscle cars: the 2018 Dodge Challenger SRT Demon and the 1970 Dodge Charger R/T. It comes with features like 2 sets of wheel rims, a rear spoiler for the Challenger, and a removable supercharger for the Charger. Plus, it includes a buildable drag racing Christmas tree start light and 3 minifigures, suitable for action-packed play.

2. **LEGO Speed Champions 1974 Porsche 911 Turbo 3.0 (75895)**
   - **Price:** $22.13
   - **Description:** This kit has 179 pieces and is perfect for Porsche enthusiasts. It features a detailed model of the classic 1974 Porsche 911 Turbo 3.0, complete with a removable windshield for placing a minifigure in the cockpit. The model includes authenti

Get the product image URL as well from retrieved products

In [ ]:
for product in response.products:
    image_url = product.get('entity').get('image_url')
    print(image_url)

https://images-na.ssl-images-amazon.com/images/I/51X5bJYgaeL.jpg
https://images-na.ssl-images-amazon.com/images/I/51FnadTjzQL.jpg
https://images-na.ssl-images-amazon.com/images/I/41flE2itPhL.jpg
